# Phase 1 — PD Motor-Balance Severity Model (H&Y Reference Classifier)

**Study design:** Train a Hoehn & Yahr severity classifier on PD patients only.  
This produces a *PD severity reference model* that learns the relationship between  
clinical balance measures and disease stage.  
The trained model is then used in Phase 2 to project welders into this learned severity space.

**Research question:**  
> Can welding-exposed workers be positioned relative to Parkinson's H&Y motor-balance  
> severity patterns — and does predicted PD-like severity track welding exposure?

**Data:** PD sheet only (n=14), `PD_WELDERS RAW Long Data-2.xlsx`  
**Validation:** Leave-One-Out Cross-Validation (LOOCV) — appropriate for n=14  
**H&Y distribution:** Stage I=2, II=3, III=6, IV=3

> ⚠ This is a pilot study (n=14 PD). All confidence intervals are wide.  
> Results are hypothesis-generating, not clinically validated.


## 0. Setup


In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn scipy openpyxl joblib -q
import pandas as pd, numpy as np, re, warnings, joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
warnings.filterwarnings('ignore')
print('Libraries loaded.')


## 1. Data Loading — PD Sheet Only

Only the PD sheet is used in Phase 1. Welders are loaded in Phase 2.


In [ ]:
# ── Upload PD_WELDERS RAW Long Data-2.xlsx when prompted (Colab), or set path ──
try:
    from google.colab import files
    print('Colab detected — please upload PD_WELDERS RAW Long Data-2.xlsx')
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
except ImportError:
    DATA_PATH = 'PD_WELDERS RAW Long Data-2.xlsx'  # local path

xl     = pd.ExcelFile(DATA_PATH)
pd_raw = xl.parse('PD')
wd_raw = xl.parse('WD')  # loaded here; used only in Phase 2

print(f'PD sheet: {pd_raw.shape[0]} rows × {pd_raw.shape[1]} columns')
print(f'WD sheet: {wd_raw.shape[0]} rows × {wd_raw.shape[1]} columns')
print(f'PD columns (balance-relevant): BBS, MINI-BEST, FES')


## 2. Helper Functions


In [ ]:
ROMAN = {'i':1, 'ii':2, 'iii':3, 'iv':4, 'v':5}

def parse_hy(val):
    """Parse H&Y stage from 'Stage I', 'Stage III', etc. → int."""
    if pd.isna(val): return np.nan
    s = str(val).lower().replace('stage','').strip()
    if s in ROMAN: return int(ROMAN[s])
    try: return int(float(re.findall(r'[\d.]+', s)[0]))
    except: return np.nan

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    """Bootstrap 95% CI on a metric from LOOCV prediction vectors."""
    rng  = np.random.RandomState(seed)
    n    = len(y_true)
    sc   = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        try: sc.append(metric_fn(np.array(y_true)[idx], np.array(y_pred)[idx]))
        except: pass
    return np.mean(sc), np.percentile(sc, 2.5), np.percentile(sc, 97.5)

def loocv_fit(clf, X, y, return_proba=False):
    """LOOCV with StandardScaler fitted per fold."""
    loo = LeaveOneOut()
    yt, yp, yprob, scalers, clfs = [], [], [], [], []
    for tr, te in loo.split(X):
        sc = StandardScaler()
        clf_copy = clone_clf(clf)
        clf_copy.fit(sc.fit_transform(X[tr]), y[tr])
        yp.append(clf_copy.predict(sc.transform(X[te]))[0])
        yt.append(y[te][0])
        if return_proba and hasattr(clf_copy, 'predict_proba'):
            yprob.append(clf_copy.predict_proba(sc.transform(X[te]))[0])
        scalers.append(sc); clfs.append(clf_copy)
    result = dict(y_true=np.array(yt), y_pred=np.array(yp),
                  scalers=scalers, clfs=clfs)
    if return_proba: result['y_proba'] = np.array(yprob)
    return result

def clone_clf(clf):
    """Return a fresh clone of a classifier."""
    from sklearn.base import clone
    return clone(clf)

def report(name, yt, yp, labels):
    acc  = accuracy_score(yt, yp)
    bal  = balanced_accuracy_score(yt, yp)
    f1   = f1_score(yt, yp, average='macro', zero_division=0)
    acc1 = np.mean(np.abs(yt - yp) <= 1)  # within-1-stage accuracy
    am, alo, ahi = bootstrap_ci(yt, yp, accuracy_score)
    fm, flo, fhi = bootstrap_ci(yt, yp,
        lambda a,b: f1_score(a, b, average='macro', zero_division=0))
    print(f'  {name}')
    print(f'    Exact Accuracy      : {acc:.3f}  95% CI [{alo:.3f}–{ahi:.3f}]')
    print(f'    Balanced Accuracy   : {bal:.3f}')
    print(f'    F1-macro            : {f1:.3f}  95% CI [{flo:.3f}–{fhi:.3f}]')
    print(f'    Within-1-stage Acc  : {acc1:.3f}')
    return dict(name=name, acc=acc, bal=bal, f1=f1, acc1=acc1,
                acc_lo=alo, acc_hi=ahi, f1_lo=flo, f1_hi=fhi)

print('Helper functions ready.')


## 3. Feature Extraction

Extract balance features and H&Y stage from the PD sheet.

**Available scales:** BBS (0–56), MINI-BEST (0–28), FES-I (10–40)  
**Note:** ABC and TUG were not collected in this dataset.


In [ ]:
records = []
for _, r in pd_raw.iterrows():
    records.append(dict(
        ID    = r.get('Participant ID', ''),
        Age   = pd.to_numeric(r.get('Age (in years)'), errors='coerce'),
        BBS   = pd.to_numeric(r.get('BBS'),            errors='coerce'),
        Mini  = pd.to_numeric(r.get('MINI-BEST'),      errors='coerce'),
        FES   = pd.to_numeric(r.get('FES'),            errors='coerce'),
        HY    = parse_hy(r.get('Current Stage of PD (Hoehn & Yahr)')),
        HY_bin= None,  # filled below
        Disease_Duration = pd.to_numeric(
            str(r.get('Disease Duration (years/months)','')).split('/')[0],
            errors='coerce'),
    ))

pd_df = pd.DataFrame(records)
pd_df['HY_bin'] = pd_df['HY'].apply(
    lambda v: 0 if pd.notna(v) and v<=2 else (1 if pd.notna(v) and v>=3 else np.nan)
)
pd_df['HY_bin_label'] = pd_df['HY_bin'].map({0:'Early (I-II)', 1:'Late (III-IV)'})

print(pd_df[['ID','Age','BBS','Mini','FES','HY','HY_bin']].to_string(index=False))
print()
print(f'H&Y distribution: {dict(pd_df.HY.value_counts().sort_index())}')
print(f'Binary: Early(I-II)={(pd_df.HY_bin==0).sum()}  Late(III-IV)={(pd_df.HY_bin==1).sum()}')
print(f'Age: {pd_df.Age.mean():.1f} ± {pd_df.Age.std(ddof=1):.1f} yrs')


## 4. Descriptive Statistics by H&Y Stage

Examine how balance scores vary across Hoehn & Yahr stages within PD.  
This motivates the classification task — if scores don't vary by stage, prediction is not feasible.


In [ ]:
print('=' * 60)
print('Balance Scores by H&Y Stage (PD n=14)')
print('=' * 60)
for col, hi_is_bad in [('BBS', False), ('Mini', False), ('FES', True)]:
    print(f'\n{col} (higher = {"worse" if hi_is_bad else "better"})')
    for stage in sorted(pd_df.HY.dropna().unique()):
        vals = pd_df[pd_df.HY==stage][col].dropna().values
        print(f'  Stage {int(stage)}: n={len(vals)}  mean={np.mean(vals):.1f}  '
              f'range=[{vals.min():.0f}–{vals.max():.0f}]')

# ── Figure: Boxplots by H&Y stage ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
palette = {1:'#3498db', 2:'#2ecc71', 3:'#f39c12', 4:'#e74c3c'}
for ax, (col, ylabel, worse) in zip(axes, [
        ('BBS',  'BBS Total (/56)',        'lower = worse'),
        ('Mini', 'Mini-BESTest (/28)',      'lower = worse'),
        ('FES',  'FES-I Total (/40)',       'higher = worse')]):
    sub = pd_df.dropna(subset=[col,'HY'])
    sns.boxplot(data=sub, x='HY', y=col, ax=ax,
                palette=palette, order=sorted(sub.HY.unique()),
                linewidth=1.2, fliersize=4)
    sns.stripplot(data=sub, x='HY', y=col, ax=ax,
                  color='black', size=5, alpha=0.7,
                  order=sorted(sub.HY.unique()), jitter=0.1)
    ax.set_xlabel('H&Y Stage'); ax.set_ylabel(ylabel)
    ax.set_title(f'{col} by H&Y Stage\n({worse})')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Balance Scores by Hoehn & Yahr Stage (PD patients, n=14)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figure1_pd_balance_by_hy.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Spearman Correlations: Balance vs H&Y Stage

Establish whether balance features have a monotonic relationship with H&Y stage  
before building a classifier. Significant correlations support predictive feasibility.


In [ ]:
print('=' * 60)
print('Spearman Correlations with H&Y Stage (PD only)')
print('=' * 60)
for col in ['BBS','Mini','FES','Age','Disease_Duration']:
    sub = pd_df.dropna(subset=[col,'HY'])
    if len(sub) < 5: continue
    rho, pval = stats.spearmanr(sub['HY'], sub[col])
    sig  = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else 'ns'
    dirn = ('↑ worse' if (col=='FES' and rho>0) or
             (col in ['BBS','Mini'] and rho<0) else
            '↓ better' if (col in ['BBS','Mini'] and rho>0) or
             (col=='FES' and rho<0) else '—')
    print(f'  {col:20s}: ρ={rho:+.3f}  p={pval:.4f} ({sig:3s})  {dirn}')
print()
print('Note: H&Y coded 1–4; negative ρ for BBS/Mini = higher stage → lower score = worse balance.')


## 6. Model A — H&Y Binary Classifier: Early (I–II) vs Late (III–IV)

**Why binary?** The 4-class H&Y model has very limited power with n=14 and  
2–6 patients per class. Binary Early vs Late is the most defensible first step.

**Features:** BBS, Mini-BESTest, FES (balance scales only — no age/disease duration  
to avoid modelling patient demographics rather than motor function).

**Validation:** LOOCV with bootstrap 95% CIs (1 000 iterations).


In [ ]:
FEAT  = ['BBS','Mini','FES']
df_bin = pd_df.dropna(subset=FEAT+['HY_bin']).reset_index(drop=True)
X_bin  = df_bin[FEAT].values.astype(float)
y_bin  = df_bin['HY_bin'].values.astype(int)

n_cls   = np.bincount(y_bin)
baseline = n_cls.max() / n_cls.sum()

print('=' * 60)
print('MODEL A — H&Y Binary: Early (I-II) vs Late (III-IV)')
print('=' * 60)
print(f'  n={len(y_bin)}  Early={n_cls[0]}  Late={n_cls[1]}')
print(f'  Majority-class baseline: {baseline:.3f}')
print()

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=300,
                                                   class_weight='balanced', random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

bin_results = {}
best_bin = {'f1': 0}
for name, clf in MODELS.items():
    res  = loocv_fit(clf, X_bin, y_bin)
    r    = report(name, res['y_true'], res['y_pred'], ['Early','Late'])
    bin_results[name] = {**r, **res}
    if r['f1'] > best_bin['f1']: best_bin = {**r, **res}
    print()

print('Best model classification report:')
print(classification_report(best_bin['y_true'], best_bin['y_pred'],
      target_names=['Early (I-II)','Late (III-IV)'], zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(5,4))
cm = confusion_matrix(best_bin['y_true'], best_bin['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Early (I-II)','Late (III-IV)']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Model A — H&Y Binary LOOCV\n{best_bin["name"]}')
plt.tight_layout()
plt.savefig('figure2_confusion_binary.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Model A2 — H&Y Multi-Class Classifier (Stages I–IV)

Train a 4-class model to learn the full H&Y spectrum within PD.  
This model is applied to welders in Phase 2 to produce stage probabilities  
for each welder.

> ⚠ With n=14 and 2–6 patients per class, exact accuracy will be low.  
> **Within-1-stage accuracy** is the more relevant metric here — it measures  
> whether predictions are at least in the adjacent stage.


In [ ]:
df_mc  = pd_df.dropna(subset=FEAT+['HY']).reset_index(drop=True)
X_mc   = df_mc[FEAT].values.astype(float)
y_mc   = df_mc['HY'].values.astype(int)
STAGES = sorted(np.unique(y_mc))
LABELS = [f'Stage {s}' for s in STAGES]

n_per_cls = dict(zip(*np.unique(y_mc, return_counts=True)))
baseline_mc = max(n_per_cls.values()) / sum(n_per_cls.values())

print('=' * 60)
print('MODEL A2 — H&Y Multi-Class (Stages I–IV)')
print('=' * 60)
print(f'  n={len(y_mc)}  distribution={n_per_cls}')
print(f'  Majority-class baseline: {baseline_mc:.3f}')
print()

mc_results  = {}
best_mc     = {'f1': 0}
for name, clf in MODELS.items():
    res = loocv_fit(clf, X_mc, y_mc)
    r   = report(name, res['y_true'], res['y_pred'], LABELS)
    mc_results[name] = {**r, **res}
    if r['f1'] > best_mc['f1']: best_mc = {**r, **res}
    print()

print('Best model classification report:')
print(classification_report(best_mc['y_true'], best_mc['y_pred'],
      target_names=LABELS, zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6,5))
cm = confusion_matrix(best_mc['y_true'], best_mc['y_pred'], labels=STAGES)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Model A2 — H&Y Multi-Class LOOCV\n{best_mc["name"]}')
plt.tight_layout()
plt.savefig('figure3_confusion_multiclass.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Save Trained Model for Phase 2

Train the best-performing binary and multi-class models on the **full PD dataset**  
(not LOOCV), then save them. Phase 2 applies these to welders.


In [ ]:
from sklearn.base import clone

# ── Binary model (full-data fit) ────────────────────────────────────────────
best_bin_clf_name = max(bin_results, key=lambda k: bin_results[k]['f1'])
best_bin_clf = clone(MODELS[best_bin_clf_name])
sc_bin = StandardScaler()
best_bin_clf.fit(sc_bin.fit_transform(X_bin), y_bin)

# ── Multi-class model (full-data fit) ───────────────────────────────────────
best_mc_clf_name = max(mc_results, key=lambda k: mc_results[k]['f1'])
best_mc_clf = clone(MODELS[best_mc_clf_name])
sc_mc = StandardScaler()
best_mc_clf.fit(sc_mc.fit_transform(X_mc), y_mc)

# ── Save ────────────────────────────────────────────────────────────────────
joblib.dump({'clf': best_bin_clf, 'scaler': sc_bin,
             'features': FEAT, 'classes': [0,1],
             'labels': ['Early (I-II)','Late (III-IV)'],
             'model_name': best_bin_clf_name},
            'pd_hy_binary_model.pkl')

joblib.dump({'clf': best_mc_clf, 'scaler': sc_mc,
             'features': FEAT, 'classes': STAGES,
             'labels': LABELS,
             'model_name': best_mc_clf_name},
            'pd_hy_multiclass_model.pkl')

print(f'Binary model saved   : pd_hy_binary_model.pkl ({best_bin_clf_name})')
print(f'Multi-class model saved: pd_hy_multiclass_model.pkl ({best_mc_clf_name})')
print()
print('These files are required by phase2_welder_projection.ipynb')


## 9. Phase 1 Results Summary


In [ ]:
print('=' * 65)
print('PHASE 1 — PD H&Y SEVERITY REFERENCE MODEL — RESULTS SUMMARY')
print('=' * 65)
print(f'  PD patients: n=14  |  H&Y: Stage I=2, II=3, III=6, IV=3')
print(f'  Features: BBS, Mini-BESTest, FES')
print(f'  Validation: LOOCV + bootstrap 95% CIs (1000 iterations)')
print()

print('MODEL A — Binary (Early I-II vs Late III-IV):')
best_b = max(bin_results.values(), key=lambda r: r['f1'])
print(f'  Best: {best_b["name"]}')
print(f'    Exact Accuracy   : {best_b["acc"]:.3f}  [{best_b["acc_lo"]:.3f}–{best_b["acc_hi"]:.3f}]')
print(f'    Balanced Accuracy: {best_b["bal"]:.3f}')
print(f'    F1-macro         : {best_b["f1"]:.3f}  [{best_b["f1_lo"]:.3f}–{best_b["f1_hi"]:.3f}]')
print(f'  Majority baseline: {baseline:.3f}')
print()

print('MODEL A2 — Multi-Class (Stages I–IV):')
best_m = max(mc_results.values(), key=lambda r: r['f1'])
print(f'  Best: {best_m["name"]}')
print(f'    Exact Accuracy   : {best_m["acc"]:.3f}  [{best_m["acc_lo"]:.3f}–{best_m["acc_hi"]:.3f}]')
print(f'    Balanced Accuracy: {best_m["bal"]:.3f}')
print(f'    F1-macro         : {best_m["f1"]:.3f}  [{best_m["f1_lo"]:.3f}–{best_m["f1_hi"]:.3f}]')
print(f'    Within-1-stage   : {best_m["acc1"]:.3f}')
print(f'  Majority baseline: {baseline_mc:.3f}')
print()

print('KEY LIMITATIONS')
print('  - n=14 PD — all CIs are wide; interpret as pilot/exploratory')
print('  - 2 patients in Stage I, 3 in Stage II — rare classes')
print('  - ABC and TUG not collected; only BBS/MiniBEST/FES available')
print('  - Cross-sectional design; disease progression not captured')
print()
print('OUTPUT FILES')
print('  pd_hy_binary_model.pkl     → used by phase2_welder_projection.ipynb')
print('  pd_hy_multiclass_model.pkl → used by phase2_welder_projection.ipynb')
print('  figure1_pd_balance_by_hy.png')
print('  figure2_confusion_binary.png')
print('  figure3_confusion_multiclass.png')
